# 11 — MURA all-anatomy preprocessing and QC

This notebook reports the local image-integrity and geometry audit that was run before feature extraction. It stores aggregate metadata only; no radiographs are displayed or saved here.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

def locate_project_root():
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "CHARTER.md").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the repository checkout.")

ROOT = locate_project_root()
RUNTIME = ROOT.parent / 'runtime' / 'multianatomy_v1'
MANIFESTS = RUNTIME / 'manifests'
QC = RUNTIME / 'qc'
BASELINE = RUNTIME / 'baseline_all_v1'
assert ROOT.name == 'mura-wrist-ml-pipeline', ROOT
print('Repository: local project checkout')
print('Runtime artifacts: local only (not committed to Git)')


Repository: local project checkout
Runtime artifacts: local only (not committed to Git)


In [2]:
qc_report = json.loads((QC / 'qc_report.json').read_text(encoding='utf-8'))
assert qc_report['official_valid_read'] is False
display(Markdown('**QC provenance:** official train only; official `valid` was not read.'))
display(pd.DataFrame([qc_report]))

**QC provenance:** official train only; official `valid` was not read.

,scope,official_valid_read,input_manifest,input_split_manifest,dataset_root,total_images,total_studies,total_decode_failures,total_low_resolution_warnings,preprocessing_contract
0,MURA official train only; all seven anatomies,False,C:\Users\netaf\Documents\Codex\2026-09-23\impo...,C:\Users\netaf\Documents\Codex\2026-09-23\impo...,C:\Users\netaf\Documents\Codex\2026-09-23\impo...,36808,13457,0,53,{'input': 'grayscale radiograph converted to t...


In [3]:
qc_summary = pd.read_csv(QC / 'qc_summary_by_anatomy.csv')
geometry = pd.read_csv(QC / 'geometry_summary_by_anatomy.csv')
display(qc_summary)
display(geometry)

,anatomy,final_split,patients,studies,images,normal_studies,abnormal_studies,decode_pass_images,decode_warning_images,decode_fail_images
0,XR_ELBOW,train,1377,1409,3962,879,530,3962,0,0
1,XR_ELBOW,val,334,345,969,215,130,969,0,0
2,XR_FINGER,train,1495,1543,4060,1017,526,4021,39,0
3,XR_FINGER,val,370,392,1046,263,129,1033,13,0
4,XR_FOREARM,train,683,694,1437,462,232,1436,1,0
5,XR_FOREARM,val,182,183,388,128,55,388,0,0
6,XR_HAND,train,1551,1607,4413,1193,414,4413,0,0
7,XR_HAND,val,394,411,1130,304,107,1130,0,0
8,XR_HUMERUS,train,471,474,1019,258,216,1019,0,0
9,XR_HUMERUS,val,116,118,253,63,55,253,0,0


,anatomy,final_split,images,width_min,width_max,height_min,height_max,aspect_ratio_p05,aspect_ratio_median,aspect_ratio_p95,source_modes
0,XR_ELBOW,train,3962,166.0,512.0,223.0,512.0,0.513672,0.792969,1.261084,L:2380;RGB:1582
1,XR_ELBOW,val,969,191.0,512.0,263.0,512.0,0.512500,0.792969,1.261084,L:572;RGB:397
2,XR_FINGER,train,4060,89.0,512.0,184.0,512.0,0.298828,0.740234,1.261084,L:2294;RGB:1766
3,XR_FINGER,val,1046,97.0,512.0,162.0,512.0,0.289062,0.740234,1.261084,L:634;RGB:412
4,XR_FOREARM,train,1437,117.0,512.0,132.0,512.0,0.380859,0.820312,2.359447,L:529;RGB:908
5,XR_FOREARM,val,388,149.0,512.0,147.0,512.0,0.390723,0.820312,2.525956,L:149;RGB:239
6,XR_HAND,train,4413,160.0,512.0,272.0,512.0,0.578125,0.792969,1.180813,L:2443;RGB:1970
7,XR_HAND,val,1130,215.0,512.0,343.0,512.0,0.582910,0.792969,1.219048,L:619;RGB:511
8,XR_HUMERUS,train,1019,174.0,512.0,176.0,512.0,0.579688,0.820312,1.219048,L:561;RGB:458
9,XR_HUMERUS,val,253,235.0,512.0,232.0,512.0,0.679297,0.820312,1.219048,L:141;RGB:112


In [4]:
plot = qc_summary.pivot(index='anatomy', columns='final_split', values='decode_fail_images').fillna(0)
ax = plot.plot(kind='bar', figsize=(10, 4.2), color=['#2563eb', '#0f766e'])
ax.set(title='Unreadable image files found by anatomy', xlabel='Anatomy', ylabel='Decode failures')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

C:\Users\netaf\AppData\Local\Temp\ipykernel_26060\3419848179.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Preprocessing contract

Images are decoded as grayscale, replicated to three channels for ImageNet ResNet-18, padded to preserve aspect ratio, resized to 224×224, and normalized with fixed ImageNet values. The evaluation transform is deterministic. Narrow but valid source images are recorded as warnings, not silently removed.